# Notebook 9: RAG vs RLM — A Hands-On Comparison

**RAG** (Retrieval-Augmented Generation) is the dominant approach for connecting LLMs to external data. How does it compare to RLMs?

In this notebook you'll learn:
- How to build a simple RAG pipeline from scratch
- How to evaluate both RAG and RLM using **RAGAS** metrics
- When RAG wins and when RLM wins
- The fundamental architectural difference: retrieval vs reasoning

## RAGAS Metrics We'll Use

| Metric | What it measures |
|--------|-----------------|
| **Faithfulness** | Is the answer grounded in the context? (no hallucinations) |
| **Answer Relevancy** | Is the answer relevant to the question? |
| **Context Precision** | Were the retrieved contexts relevant? |
| **Context Recall** | Were all relevant contexts found? |

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
import json, os

from rlm_core import RLMEngine, Sandbox
from rlm_core.visualizer import tree_to_text

# Simulated client for demos
class SimulatedClient:
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split())
        self.total_completion_tokens += 15
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text, self.prompt_tokens, self.completion_tokens = text, pt, ct
        
        if "secret" in prompt.lower() or "code" in prompt.lower():
            text = '''import re
match = re.search(r'secret code is ([A-Z0-9-]+)', context)
FINAL(match.group(1) if match else "Not found")'''
        elif "how many" in prompt.lower() or "count" in prompt.lower() or "red" in prompt.lower():
            text = '''import json
data = json.loads(context) if context.strip().startswith('[') else []
count = sum(1 for i in data if i.get("color")=="red" and i.get("size")=="large")
FINAL(f"{count}")'''
        elif "engineer" in prompt.lower() or "who" in prompt.lower():
            text = '''import re
m = re.search(r'(Dr\\.? [A-Z][a-z]+ [A-Z][a-z]+|[A-Z][a-z]+ [A-Z][a-z]+).*(?:lead engineer|CTO)', context)
FINAL(m.group(1) if m else "Not found")'''
        else:
            text = 'FINAL("Information processed")'
        
        return Result(text, len(prompt.split()), 15)

client = SimulatedClient()
engine = RLMEngine(client=client, max_depth=3)

# Load all datasets
with open("../data/samples/needle_haystack.txt") as f:
    needle_doc = f.read()
with open("../data/samples/aggregation_items.json") as f:
    agg_items = json.load(f)
docs = []
doc_dir = "../data/samples/multihop_docs"
for fname in sorted(os.listdir(doc_dir)):
    if fname.endswith('.txt'):
        with open(os.path.join(doc_dir, fname)) as f:
            docs.append(f.read())

print(f"Loaded: needle ({len(needle_doc)} chars), {len(agg_items)} items, {len(docs)} docs")

## What is RAG?

RAG works in 5 steps:
```
Document → Chunk → Embed → Store → Retrieve → Generate
```

1. **Chunk**: Split documents into small pieces (~200-500 tokens each)
2. **Embed**: Convert each chunk into a vector (numerical representation)
3. **Store**: Save vectors in a vector database
4. **Retrieve**: Find the top-k most similar chunks to the user's question
5. **Generate**: Feed retrieved chunks to the LLM as context

## Step 1: Build a Simple RAG Pipeline

We'll build RAG from scratch using basic Python (no external vector DB needed for this demo):

In [ ]:
import re
from collections import Counter
import math

class SimpleRAG:
    """A minimal RAG implementation using TF-IDF for retrieval."""
    
    def __init__(self, chunk_size=200):
        self.chunk_size = chunk_size
        self.chunks = []
        self.chunk_vectors = []  # TF-IDF vectors
        self.vocab = {}
        self.idf = {}
    
    def add_documents(self, documents):
        """Chunk and index documents."""
        for doc in documents:
            words = doc.split()
            for i in range(0, len(words), self.chunk_size):
                chunk = " ".join(words[i:i + self.chunk_size])
                self.chunks.append(chunk)
        
        # Build TF-IDF
        self._build_tfidf()
        print(f"Indexed {len(self.chunks)} chunks from {len(documents)} document(s)")
    
    def _build_tfidf(self):
        """Build TF-IDF vectors for all chunks."""
        # Build vocabulary
        all_words = set()
        for chunk in self.chunks:
            all_words.update(re.findall(r'\w+', chunk.lower()))
        self.vocab = {w: i for i, w in enumerate(sorted(all_words))}
        
        # Compute IDF
        doc_freq = Counter()
        for chunk in self.chunks:
            words = set(re.findall(r'\w+', chunk.lower()))
            for w in words:
                doc_freq[w] += 1
        
        n = len(self.chunks)
        self.idf = {w: math.log(n / (1 + df)) for w, df in doc_freq.items()}
        
        # Compute TF-IDF vectors
        self.chunk_vectors = []
        for chunk in self.chunks:
            words = re.findall(r'\w+', chunk.lower())
            tf = Counter(words)
            vector = {}
            for w, count in tf.items():
                if w in self.idf:
                    vector[w] = (count / len(words)) * self.idf[w]
            self.chunk_vectors.append(vector)
    
    def retrieve(self, query, top_k=3):
        """Find the top-k most relevant chunks for a query."""
        query_words = re.findall(r'\w+', query.lower())
        query_tf = Counter(query_words)
        query_vector = {}
        for w, count in query_tf.items():
            if w in self.idf:
                query_vector[w] = (count / len(query_words)) * self.idf[w]
        
        # Cosine similarity
        scores = []
        for i, chunk_vec in enumerate(self.chunk_vectors):
            dot = sum(query_vector.get(w, 0) * chunk_vec.get(w, 0) for w in query_vector)
            norm_q = math.sqrt(sum(v**2 for v in query_vector.values())) + 1e-10
            norm_c = math.sqrt(sum(v**2 for v in chunk_vec.values())) + 1e-10
            scores.append((dot / (norm_q * norm_c), i))
        
        scores.sort(reverse=True)
        return [(self.chunks[i], score) for score, i in scores[:top_k]]
    
    def query(self, question, client, top_k=3):
        """Full RAG pipeline: retrieve + generate."""
        retrieved = self.retrieve(question, top_k)
        
        context = "\n\n".join(f"[Chunk {i+1}]: {chunk[:300]}" for i, (chunk, _) in enumerate(retrieved))
        
        prompt = f"""Answer based ONLY on the provided context:

{context}

Question: {question}
Answer:"""
        
        result = client.completion(prompt)
        
        return {
            "answer": result.text,
            "retrieved_chunks": retrieved,
            "n_chunks_total": len(self.chunks),
            "n_retrieved": top_k,
        }

rag = SimpleRAG(chunk_size=100)
print("SimpleRAG ready!")

## Step 2: RAG vs RLM — Needle-in-a-Haystack

Can RAG find a hidden fact? Can RLM?

In [ ]:
# Index the needle document
rag_needle = SimpleRAG(chunk_size=100)
rag_needle.add_documents([needle_doc])

# RAG attempt
print("=== RAG: Needle-in-a-Haystack ===")
rag_result = rag_needle.query("What is the secret code?", client)
print(f"Answer: {rag_result['answer'][:100]}")
print(f"Retrieved {rag_result['n_retrieved']} of {rag_result['n_chunks_total']} chunks")
for i, (chunk, score) in enumerate(rag_result['retrieved_chunks']):
    has_needle = "BLUE-FALCON" in chunk
    print(f"  Chunk {i+1} (score: {score:.3f}): {'CONTAINS NEEDLE!' if has_needle else chunk[:60] + '...'}")

# RLM attempt
print(f"\n=== RLM: Needle-in-a-Haystack ===")
rlm_result = engine.run("What is the secret code?", needle_doc)
print(f"Answer: {rlm_result.answer}")
print(f"LLM calls: {rlm_result.total_llm_calls}")

## Step 3: RAG vs RLM — Aggregation

RAG retrieves top-k chunks. But what if you need ALL chunks to answer the question (e.g., "count all red items")?

In [ ]:
# Index aggregation data
agg_text = json.dumps(agg_items, indent=2)
rag_agg = SimpleRAG(chunk_size=50)
rag_agg.add_documents([agg_text])

print("=== RAG: Aggregation ===")
rag_agg_result = rag_agg.query("How many items are red and large?", client, top_k=3)
print(f"Answer: {rag_agg_result['answer'][:100]}")
print(f"Retrieved {rag_agg_result['n_retrieved']} of {rag_agg_result['n_chunks_total']} chunks")
print("Problem: RAG only sees 3 chunks — can't count items in chunks it didn't retrieve!")

print(f"\n=== RLM: Aggregation ===")
rlm_agg = engine.run("How many items are both red AND large?", agg_text)
print(f"Answer: {rlm_agg.answer}")
ground_truth = len([i for i in agg_items if i["color"] == "red" and i["size"] == "large"])
print(f"Ground truth: {ground_truth}")

## Step 4: RAG vs RLM — Multi-hop QA

Multi-hop questions require connecting facts across documents. RAG needs to retrieve ALL relevant docs.

In [ ]:
# Index multi-hop documents
rag_multi = SimpleRAG(chunk_size=100)
rag_multi.add_documents(docs)

print("=== RAG: Multi-hop QA ===")
rag_multi_result = rag_multi.query(
    "Who is the lead engineer at the company that made the NovaPad?", client, top_k=3
)
print(f"Answer: {rag_multi_result['answer'][:100]}")
print(f"Retrieved chunks:")
for i, (chunk, score) in enumerate(rag_multi_result['retrieved_chunks']):
    print(f"  Chunk {i+1} (score: {score:.3f}): {chunk[:80]}...")
print("RAG needs both the NovaPad doc AND the lead engineer doc — did it get both?")

print(f"\n=== RLM: Multi-hop QA ===")
multi_context = "\n\n---\n\n".join(docs)
rlm_multi = engine.run("Who is the lead engineer at the company that made the NovaPad?", multi_context)
print(f"Answer: {rlm_multi.answer}")
print(f"Expected: Dr. James Park")

## Step 5: RAGAS-Style Evaluation

RAGAS provides standardized metrics for evaluating both RAG and RLM. Let's implement simplified versions:

In [ ]:
def evaluate_faithfulness(answer, context):
    """Is the answer grounded in the context? (simplified)"""
    answer_words = set(re.findall(r'\w+', answer.lower()))
    context_words = set(re.findall(r'\w+', context.lower()))
    if not answer_words:
        return 0.0
    grounded = answer_words & context_words
    return len(grounded) / len(answer_words)

def evaluate_answer_relevancy(answer, question):
    """Is the answer relevant to the question? (simplified)"""
    q_words = set(re.findall(r'\w+', question.lower()))
    a_words = set(re.findall(r'\w+', answer.lower()))
    if not q_words:
        return 0.0
    overlap = q_words & a_words
    return len(overlap) / len(q_words)

def evaluate_context_precision(retrieved_chunks, ground_truth_answer):
    """Were the retrieved contexts relevant? (contained the answer?)"""
    if not retrieved_chunks:
        return 0.0
    relevant = sum(1 for chunk, _ in retrieved_chunks 
                   if ground_truth_answer.lower() in chunk.lower())
    return relevant / len(retrieved_chunks)

def evaluate_context_recall(retrieved_chunks, ground_truth_context_keywords):
    """Were all relevant pieces of context found?"""
    if not ground_truth_context_keywords:
        return 0.0
    found = sum(1 for kw in ground_truth_context_keywords
                if any(kw.lower() in chunk.lower() for chunk, _ in retrieved_chunks))
    return found / len(ground_truth_context_keywords)

def ragas_evaluate(answer, question, context, retrieved_chunks=None, 
                   ground_truth="", gt_keywords=None):
    """Run all RAGAS metrics."""
    results = {
        "faithfulness": evaluate_faithfulness(answer, context),
        "answer_relevancy": evaluate_answer_relevancy(answer, question),
    }
    if retrieved_chunks is not None:
        results["context_precision"] = evaluate_context_precision(retrieved_chunks, ground_truth)
        results["context_recall"] = evaluate_context_recall(retrieved_chunks, gt_keywords or [])
    return results

print("RAGAS evaluation functions ready!")

## Step 6: RAGAS Results — RAG vs RLM

In [ ]:
import matplotlib.pyplot as plt

# Evaluate needle task
needle_q = "What is the secret code?"
needle_gt = "BLUE-FALCON-42"
needle_keywords = ["secret", "code", "BLUE-FALCON-42"]

rag_scores = ragas_evaluate(
    answer=str(rag_result['answer']),
    question=needle_q,
    context=needle_doc,
    retrieved_chunks=rag_result['retrieved_chunks'],
    ground_truth=needle_gt,
    gt_keywords=needle_keywords
)

rlm_scores = ragas_evaluate(
    answer=str(rlm_result.answer),
    question=needle_q,
    context=needle_doc,
    ground_truth=needle_gt,
)

print("=== RAGAS Scores: Needle-in-a-Haystack ===")
print(f"{'Metric':<25} {'RAG':>8} {'RLM':>8}")
print("-" * 45)
for metric in ['faithfulness', 'answer_relevancy']:
    print(f"{metric:<25} {rag_scores[metric]:>8.2f} {rlm_scores[metric]:>8.2f}")
if 'context_precision' in rag_scores:
    print(f"{'context_precision':<25} {rag_scores['context_precision']:>8.2f} {'N/A':>8}")
    print(f"{'context_recall':<25} {rag_scores['context_recall']:>8.2f} {'N/A':>8}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['faithfulness', 'answer_relevancy']
rag_vals = [rag_scores[m] for m in metrics]
rlm_vals = [rlm_scores[m] for m in metrics]

x = range(len(metrics))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], rag_vals, width, label='RAG', color='#FF6B6B')
bars2 = ax.bar([i + width/2 for i in x], rlm_vals, width, label='RLM', color='#4ECDC4')

ax.set_ylabel('Score (0-1)')
ax.set_title('RAGAS Evaluation: RAG vs RLM (Needle-in-Haystack)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## When RAG Wins vs When RLM Wins

| Scenario | RAG | RLM | Why |
|----------|-----|-----|-----|
| Simple fact retrieval | Wins | Comparable | RAG is faster and cheaper |
| Aggregation (count/sum) | Loses | Wins | RAG only sees top-k chunks |
| Multi-hop reasoning | Loses | Wins | RAG may not retrieve all needed docs |
| Very large corpus | Wins | Loses | RAG scales with vector search; RLM must process all text |
| Real-time / low latency | Wins | Loses | RAG is one retrieval + one LLM call |

**The fundamental difference:**
- **RAG** asks: "Which chunks are SIMILAR to the question?" (retrieval)
- **RLM** asks: "How should I PROCESS this data to find the answer?" (reasoning)

## Exercise

Create a task where RAG fails but RLM succeeds (e.g., "What percentage of items are in stock?").

In [ ]:
# Exercise: RAG fails on aggregation tasks

print("=== Your Exercise: Percentage Calculation ===")
question = "What percentage of items are in stock?"

# RAG attempt
rag_exercise = SimpleRAG(chunk_size=50)
rag_exercise.add_documents([json.dumps(agg_items)])
rag_ex_result = rag_exercise.query(question, client, top_k=3)
print(f"RAG answer: {rag_ex_result['answer'][:100]}")
print(f"RAG saw {rag_ex_result['n_retrieved']} of {rag_ex_result['n_chunks_total']} chunks")

# Ground truth
in_stock = sum(1 for i in agg_items if i.get("in_stock", False))
print(f"\nGround truth: {in_stock}/{len(agg_items)} = {in_stock/len(agg_items)*100:.0f}%")
print("RAG can't answer this correctly — it only sees a subset of the data!")

## Key Takeaways

1. **RAG retrieves, RLM reasons** — fundamentally different approaches
2. **RAG excels** at finding relevant passages in large corpora quickly and cheaply
3. **RLM excels** at tasks requiring aggregation, multi-hop reasoning, or full-context processing
4. **RAGAS metrics** (faithfulness, relevancy, precision, recall) provide standardized evaluation
5. **They're complementary** — use RAG for retrieval tasks, RLM for reasoning tasks
6. **Context precision/recall** highlight RAG's fundamental limitation: it can only use what it retrieves

## What's Next?

Notebook 10 compares RLM vs **ReAct agents** — the dominant agent paradigm.